# Guide 23: Distribution System Loss Analysis

Distribution losses represent energy that is paid for but never delivered to customers.
Understanding where losses occur helps prioritize infrastructure investments.

**What you will learn:**
- System loss breakdown by element type
- Identifying top loss contributors
- How losses vary with load (I²R relationship)
- Seasonal loss patterns from time-series simulation


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

RESULTS = '../sisyphean-power-and-light/network/results'


## System Overview


In [ ]:
with open(f'{RESULTS}/summary.json') as f:
    s = json.load(f)

print(f'Total load:      {s["total_power_kw"]:,.0f} kW')
print(f'Total losses:    {s["total_loss_kw"]:,.0f} kW')
print(f'Loss rate:       {s["loss_pct"]:.2f}%')
print(f'Reactive losses: {s["total_loss_kvar"]:,.0f} kVAR')


## Losses by Element Type

Lines (conductors) dominate distribution losses due to I²R heating.
Transformer losses include load-dependent copper losses and fixed core losses.


In [ ]:
losses = pd.read_csv(f'{RESULTS}/system_losses.csv')
by_type = losses.groupby('element_type').agg(
    total_kw=('kw_loss', 'sum'),
    total_kvar=('kvar_loss', 'sum'),
    count=('element_name', 'count'),
).sort_values('total_kw', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
by_type['total_kw'].plot.bar(ax=ax, color='#1C4855')
ax.set_ylabel('Total Losses (kW)')
ax.set_title('Distribution Losses by Element Type')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

total = losses['kw_loss'].sum()
for t, row in by_type.iterrows():
    print(f'{t:15s}: {row["total_kw"]:8.1f} kW ({row["total_kw"]/total*100:5.1f}%) across {row["count"]} elements')


## Top Loss Contributors


In [ ]:
top = losses.nlargest(20, 'kw_loss')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1C4855' if 'Line' in n else '#5FCCDB' for n in top['element_name']]
ax.barh(range(len(top)), top['kw_loss'], color=colors)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top['element_name'], fontsize=8)
ax.set_xlabel('Loss (kW)')
ax.set_title('Top 20 Loss Contributors (dark=Line, light=Transformer)')
plt.tight_layout()
plt.show()

top_share = top['kw_loss'].sum() / total * 100
print(f'Top 20 elements: {top["kw_loss"].sum():.1f} kW ({top_share:.1f}% of total)')


## Loss vs Loading

Losses scale with the square of current (I²R). Heavily loaded lines
contribute disproportionately to system losses.


In [ ]:
line_flows = pd.read_parquet(f'{RESULTS}/line_flows.parquet')
line_losses = losses[losses['element_type'] == 'Line'].copy()
line_losses['short_name'] = line_losses['element_name'].str.replace('Line.', '', regex=False)
line_flows['short_name'] = line_flows['line_name']

merged = line_flows.merge(line_losses, on='short_name', how='inner')

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(merged['loading_pct'], merged['kw_loss'], alpha=0.4, c='#1C4855', s=15)
ax.set_xlabel('Line Loading (%)')
ax.set_ylabel('Line Loss (kW)')
ax.set_title('Loss vs Loading (I²R relationship)')
plt.tight_layout()
plt.show()


## Seasonal Loss Variation

Time-series simulation (QSTS) reveals how losses track load throughout the day and across seasons.
Summer peaks drive the highest losses due to the quadratic I²R relationship.


In [ ]:
ts = pd.read_parquet(f'{RESULTS}/qsts_timeseries.parquet')
ts['timestamp'] = pd.to_datetime(ts['timestamp'])

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

colors = {'winter': '#4A90D9', 'spring': '#50C878', 'summer': '#E74C3C', 'fall': '#E7A33E'}
for season in ['winter', 'spring', 'summer', 'fall']:
    sd = ts[ts['season'] == season]
    axes[0].plot(sd['hour'], sd['source_kw'], label=season.capitalize(),
                color=colors[season], alpha=0.8)
    axes[1].plot(sd['hour'], sd['loss_kw'], label=season.capitalize(),
                color=colors[season], alpha=0.8)

axes[0].set_ylabel('Source Power (kW)')
axes[0].set_title('Load Profile by Season')
axes[0].legend()
axes[1].set_ylabel('Losses (kW)')
axes[1].set_xlabel('Hour of Week')
axes[1].set_title('Loss Profile by Season')
axes[1].legend()

plt.tight_layout()
plt.show()

for season in ['winter', 'spring', 'summer', 'fall']:
    sd = ts[ts['season'] == season]
    print(f'{season:8s}: avg load {sd["source_kw"].mean():,.0f} kW, '
          f'avg loss {sd["loss_kw"].mean():.0f} kW '
          f'({sd["loss_kw"].mean()/sd["source_kw"].mean()*100:.1f}%)')


## Key Takeaways

1. **Lines dominate losses**: Conductor I²R heating is the primary loss mechanism in distribution.
2. **Losses scale quadratically**: A 2x increase in load produces ~4x increase in losses.
3. **Top contributors**: A small number of heavily loaded trunk lines drive most losses.
4. **Seasonal pattern**: Summer peak losses are 3-5x higher than spring minimum.
5. **Reduction strategies**: Capacitor banks reduce reactive current; conductor upgrades reduce resistance.

See Guide 21 for power flow fundamentals and Guide 22 for hosting capacity analysis.
